In [1]:
## Imports
import numpy as np
import torch
import json
import os
from tabulate import tabulate
from PIL import Image
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import ImageNet, CIFAR10, CIFAR100, ImageFolder
from utils.misc.misc import accuracy, accuracy_correct
from utils.scripts.algorithms_text_explanations import svd_data_approx
from utils.scripts.algorithms_text_explanations_funcs import *
from utils.models.factory import create_model_and_transforms, get_tokenizer
from utils.models.prs_hook_text import hook_prs_logger_text
from utils.misc.visualization import visualization_preprocess
from utils.datasets_constants.imagenet_classes import imagenet_classes
from utils.datasets.dataset_helpers import dataset_subset


/home/ggil/anaconda3/envs/MT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Parameters
device = 'cpu'
model_name = 'ViT-B-32'        # 'ViT-H-14', 'ViT-L-14', 'ViT-B-16', 'ViT-B-32'
seed = 0
num_last_layers_ = 4           # how many of the last TEXT-encoder layers to decompose
algorithm = "svd_data_approx"
# Text decomposed into components (run through the text encoder, gathered at EOS).
dataset = "imagenet_descriptions_personal"
# imagenet_descriptions_personal has 10 sentences/class in class order; the LAST is the class name.
native_per_class = 10          # sentences per class in `dataset` (1 = no class structure)
sentences_per_class = 1        # keep the LAST k per class; 1 -> only the class-name sentence
# Text bank used to label the components (text-side characterization).
dataset_text_name = "top_1500_nouns_5_sentences_imagenet_clean"
# Image set used to charachterize the components (image-side characterization) + visualization.
datataset_image_name = "imagenet"
subset_dim = 10
tot_samples_per_class = 50
path = './datasets/'
cache_dir = "../cache"

if model_name == "ViT-H-14":
    pretrained = "laion2B-s32B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14":
    pretrained = "laion2B-s32B-b82K"; precision = "fp32"
elif model_name == "ViT-B-16":
    pretrained = "laion2B-s34B-b88K"; precision = "fp32"
elif model_name == "ViT-B-32":
    pretrained = "laion2B-s34B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14-336":
    pretrained = "openai"; precision = "fp16"


In [5]:
## Loading Model
model, _, preprocess = create_model_and_transforms(
    model_name, pretrained=pretrained, precision=precision, cache_dir=cache_dir)
model.to(device)
model.eval()
tokenizer = get_tokenizer(model_name)

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Number of TEXT-encoder layers:", len(model.transformer.resblocks))

Using local files
Model parameters: 151,277,313
Number of TEXT-encoder layers: 12


In [ ]:
## Run the spectral decomposition on the text-encoder heads (text explanations)
command = (f"python -m utils.scripts.compute_text_explanations_text "
           f"--device {device} --model {model_name} --algorithm {algorithm} --seed {seed} "
           f"--text_per_princ_comp 20 --num_of_last_layers {num_last_layers_} "
           f"--bank {dataset} --text_descriptions {dataset_text_name}")
!{command}


In [ ]:
# Load the text-encoder component datasets
attention_dataset = f"output_dir/{dataset}_completeness_text_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

# Text-encoder components (decomposition of the bank at the EOS token)
attns_ = torch.tensor(np.load(f"output_dir/{dataset}_attn_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_  = torch.tensor(np.load(f"output_dir/{dataset}_mlp_text_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]

# Probes: text-side (bank embeddings, also the reconstruction target) and image-side (ImageNet)
final_embeddings_texts  = torch.tensor(np.load(f"output_dir/{dataset_text_name}_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
final_embeddings_images = torch.tensor(np.load(f"output_dir/{datataset_image_name}_embeddings_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)

# Probe-side text labels (aligned with final_embeddings_texts)
with open(f"utils/text_descriptions/{dataset_text_name}.txt", "r") as f:
    texts_str = np.array([i.replace("\n", "") for i in f.readlines()])

# Decomposed-corpus texts, aligned with attns_ rows (same last-k-per-class subsampling as prepare)
with open(f"utils/text_descriptions/{dataset}.txt", "r") as f:
    _all = [i.replace("\n", "") for i in f.readlines()]
corpus_texts = np.array([l for c in range(len(_all) // native_per_class)
                         for l in _all[(c + 1) * native_per_class - sentences_per_class:(c + 1) * native_per_class]])

# Class labels (row -> ImageNet class) and the class-prompt classifier, for accuracy
labels_ = torch.tensor(np.load(f"output_dir/{dataset}_labels_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()
classifier_ = torch.tensor(np.load(f"output_dir/{datataset_image_name}_classifier_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
image_labels_ = torch.tensor(np.load(f"output_dir/{datataset_image_name}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()  # class per image (aligned to final_embeddings_images)

# Exact decomposition target: M_text = sum_l sum_h attn + sum_l mlp  (EOS, projected)
final_embeddings_texts_decomp = attns_.sum(axis=(1, 2)) + mlps_.sum(axis=1)  # [N, d]

# Derived quantities (mirrors playground.ipynb)
no_heads_attentions_ = attns_.sum(axis=2)        # [N, l, d], summed over heads
nr_layers_ = attns_.shape[1]
nr_heads_  = attns_.shape[2]
last_ = attns_.shape[1] - num_last_layers_
current_mean_ablation_per_head_sum_ = torch.mean(no_heads_attentions_[:, :last_ + 1], axis=0).sum(0)

# Image dataset for the image-side characterization / visualization
ds_ = ImageNet(root=path + "imagenet/", split="val", transform=visualization_preprocess)
classes_ = imagenet_classes
ds_vis_ = dataset_subset(ds_, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class, seed=seed)

# Mean rank of the heads' PC decomposition
data = get_data(attention_dataset, skip_final=True)
mean_rank_ = sum(e["rank"] for e in data) / len(data)
print("attns", tuple(attns_.shape), "mlps", tuple(mlps_.shape), "| mean head rank:", round(mean_rank_, 2))


# Top principal-component text interpretation for each text-encoder head

In [ ]:
data = get_data(attention_dataset, -1)
print_data(data, 4)

# Strongest principal components across the text-encoder

In [ ]:
data = get_data(attention_dataset, -1, skip_final=True)
top_k_entries = top_data(sort_data_by(data, "strength_abs", descending=True), top_k=10)
print_data(top_k_entries)

# Characterize one component bidirectionally (nearest images AND nearest texts)
`visualize_principal_component` scores the PC direction `vh` against both the image probes
and the text probes, so each text-encoder component is shown from both modalities.

In [ ]:
layer = nr_layers_ - 1
head = 0
princ_comp = 0
nr_top_imgs = 20
nr_worst_imgs = 20
nr_cont_imgs = 0

In [ ]:
visualize_principal_component(layer, head, princ_comp, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
    attention_dataset, final_embeddings_images, final_embeddings_texts, seed, path, texts_str,
    dataset=datataset_image_name, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class)

# Per-PC cosine similarity to images vs. texts, across the last layers

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns


def visualize_pc_cosine_similarity(
    data,
    final_embeddings_images: torch.Tensor,
    final_embeddings_texts: torch.Tensor,
    base_layer: int,
    model_name: str = "ViT-B-32",
    max_pc: int = 10,
    top_scores: int = 5,
    n_permutations: int = 10
):
    # ---- sorted ascending layers ----
    layers = sorted(base_layer - i for i in range(4))

    # ---- normalize embeddings ----
    texts_norm  = final_embeddings_texts  / final_embeddings_texts.norm(dim=-1, keepdim=True)
    images_norm = final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)

    # ---- allocate arrays ----
    cos_mean_text = np.zeros((max_pc, len(layers)))
    cos_std_text  = np.zeros_like(cos_mean_text)
    cos_mean_img  = np.zeros_like(cos_mean_text)
    cos_std_img   = np.zeros_like(cos_mean_text)

    # ---- fill per‑PC, per‑layer statistics ----
    for j, layer in enumerate(layers):
        for i in range(max_pc):
            e = [e for e in data if e['layer'] == layer and e['princ_comp'] == i][0]
            vh     = torch.tensor(e['vh'])
            pc_dir = vh[i] / vh[i].norm()

            txt_cos = torch.topk(torch.abs(texts_norm @ pc_dir), top_scores).values
            img_cos = torch.topk(torch.abs(images_norm @ pc_dir), top_scores).values

            cos_mean_text[i, j] = txt_cos.mean().item()
            cos_std_text[i, j]  = txt_cos.std().item()
            cos_mean_img[i, j]  = img_cos.mean().item()
            cos_std_img[i, j]   = img_cos.std().item()

    # ---- build annotation strings “mean±std” ----
    ann_text = np.array([[f"{m:.2f}±{s:.2f}" for m, s in zip(row_m, row_s)]
                         for row_m, row_s in zip(cos_mean_text, cos_std_text)])
    ann_img = np.array([[f"{m:.2f}±{s:.2f}" for m, s in zip(row_m, row_s)]
                        for row_m, row_s in zip(cos_mean_img, cos_std_img)])

    # ---- determine shared color scale ----
    vmin = min(cos_mean_text.min(), cos_mean_img.min())
    vmax = max(cos_mean_text.max(), cos_mean_img.max())

    # ---- plot heatmaps ----
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, max_pc * 0.6 + 3),
        gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.3}
    )
    fig.suptitle(f"Model: {model_name} — PC Cosine Similarities", fontsize=18, y=0.99)

    # first heatmap without individual colorbar
    sns.heatmap(
        cos_mean_text,
        annot=ann_text,
        fmt="",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        ax=axes[0],
        yticklabels=[f"PC {i}" for i in range(max_pc)],
        xticklabels=[f"Layer {l}" for l in layers]
    )
    axes[0].set_title("Top Text Cosine Similarity", fontsize=14)

    # second heatmap without individual colorbar
    sns.heatmap(
        cos_mean_img,
        annot=ann_img,
        fmt="",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        ax=axes[1],
        yticklabels=[f"PC {i}" for i in range(max_pc)],
        xticklabels=[f"Layer {l}" for l in layers]
    )
    axes[1].set_title("Top Image Cosine Similarity", fontsize=14)

    # add a single colorbar to the right
    cbar_ax = fig.add_axes([0.92, 0.3, 0.02, 0.4])  # [left, bottom, width, height]
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label="Cosine Similarity")

    # ---- compute summary stats ----
    pairwise     = (texts_norm @ images_norm.T).cpu().numpy()
    global_mean  = pairwise.mean()
    global_std   = pairwise.std()

    # random permutations
    n_txt, n_img = texts_norm.size(0), images_norm.size(0)
    pairs        = min(n_txt, n_img)
    txt_txt, img_img, txt_img = [], [], []

    for _ in range(n_permutations):
        p_t = torch.randperm(n_txt)
        txt_txt.append(((texts_norm * texts_norm[p_t]).sum(dim=1)).mean().item())

        p_i = torch.randperm(n_img)
        img_img.append(((images_norm * images_norm[p_i]).sum(dim=1)).mean().item())

        idx_t = torch.randperm(n_txt)[:pairs]
        idx_i = torch.randperm(n_img)[:pairs]
        txt_img.append(((texts_norm[idx_t] * images_norm[idx_i]).sum(dim=1)).mean().item())

    tt_mean, tt_std = np.mean(txt_txt),  np.std(txt_txt)
    ii_mean, ii_std = np.mean(img_img),  np.std(img_img)
    ti_mean, ti_std = np.mean(txt_img),  np.std(txt_img)

    # ---- add summary box below ----
    summary = (
        f"Text–Image: {global_mean:.2f}±{global_std:.2f}    "
        f"Text–Text (Random): {tt_mean:.2f}±{tt_std:.3f}\n"
        f"Image–Image (Random): {ii_mean:.2f}±{ii_std:.3f}    "
    )
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    fig.text(
        0.5,
        0,
        summary,
        ha='center',
        va='center',
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray")
    )
    plt.show()


In [ ]:
visualize_pc_cosine_similarity(data, final_embeddings_images, final_embeddings_texts,
    nr_layers_ - 1, model_name=model_name, max_pc=5)

# Singular-value (variance) profiles of the components

In [ ]:
plot_pc_sv_across_heads_and_layers(data, nr_layers_ - 1)

In [ ]:
data = get_data(attention_dataset)
plot_pc_sv(data, layer, head)

# Query a topic / image and find nearest neighbours via the text components

In [ ]:
## Query a topic (text or image) and reconstruct it from the TEXT-encoder components
model.eval()
query_text = True
max_pcs_per_head = 10

with torch.no_grad():
    if query_text:
        text_query = "An image of colour black."
        topic_emb = model.encode_text(tokenizer(text_query).to(device), normalize=True).to(torch.float32)
    else:
        text_query = "woman.png"
        image = preprocess(Image.open(f'images/{text_query}'))[np.newaxis].to(device)
        if precision == "fp16":
            image = image.to(dtype=torch.float16)
        topic_emb = model.encode_image(image, attn_method='head_no_spatial', normalize=True).to(torch.float32)

# Which text-encoder principal components reconstruct the query?
data = get_data(attention_dataset, max_pcs_per_head, skip_final=True)
mean_final_images = torch.mean(final_embeddings_images, axis=0).to(device)
mean_final_texts  = torch.mean(final_embeddings_texts,  axis=0).to(device)
mean_final = mean_final_texts if query_text else mean_final_images

topic_emb_cent = topic_emb - mean_final
[topic_emb_rec_cent], data = reconstruct_embeddings(
    data, [topic_emb_cent], ["text" if query_text else "image"],
    return_princ_comp=True, plot=True, means=[mean_final], device=device)

topic_emb_rec_cent_norm = topic_emb_rec_cent / topic_emb_rec_cent.norm(dim=-1, keepdim=True)
topic_emb_cent_norm = topic_emb_cent / topic_emb_cent.norm(dim=-1, keepdim=True)
max_reconstr_score = topic_emb_rec_cent_norm @ topic_emb_cent_norm.T
print(f"Max cosine similarity of the reconstruction: {max_reconstr_score.item():.4f}")


In [ ]:
# Greedy top-k text-encoder components that reconstruct the query
top_k = 30
approx = 1.1

data = sort_data_by(data, "correlation_princ_comp_abs", descending=True)
top_k_entries = top_data(data, top_k)
top_k_details = reconstruct_top_embedding(
    top_k_entries, topic_emb_cent, mean_final, "text" if query_text else "image",
    max_reconstr_score, top_k, approx, device=device)
print(f"Querying: {text_query}")
print_data(top_k_details, is_corr_present=True)


In [ ]:
## Bidirectional retrieval: nearest bank texts AND nearest images for the query
nr_top_imgs, nr_worst_imgs, nr_cont_imgs = 20, 20, 0

# Reconstruct the bank-text embeddings emphasising the query's components
texts_rec = reconstruct_all_embeddings_mean_ablation_pcs(
    top_k_entries, mlps_, attns_, attns_, nr_layers_, nr_heads_, last_,
    ratio=-1, ablation=True, mean_ablate_all=True).to(device)

# Text-side scores (index into texts_str)
scores_txt = np.empty(texts_rec.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('txt_index', 'i4')])
tr = texts_rec / texts_rec.norm(dim=-1, keepdim=True)
scores_txt['score']     = (texts_rec @ topic_emb.T).squeeze().cpu().numpy()
scores_txt['score_vis'] = (tr @ topic_emb.T).squeeze().cpu().numpy()
scores_txt['txt_index'] = np.arange(texts_rec.shape[0])

# Image-side scores (index into ds_vis_ / final_embeddings_images)
scores_img = np.empty(final_embeddings_images.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('img_index', 'i4')])
fi = final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)
scores_img['score']     = (final_embeddings_images @ topic_emb.T).squeeze().cpu().numpy()
scores_img['score_vis'] = (fi @ topic_emb.T).squeeze().cpu().numpy()
scores_img['img_index'] = np.arange(final_embeddings_images.shape[0])

dbs = create_dbs(scores_img, scores_txt, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
                 text_query=text_query, max_reconstr_score=max_reconstr_score)
dbs = [db for nr, db in zip([nr_top_imgs, nr_worst_imgs, nr_cont_imgs], dbs) if nr > 0]
visualize_dbs(top_k_details, dbs, ds_vis_, corpus_texts, classes_, text_query)


# Ablation study

In [ ]:
## EOS reconstruction error under mean-ablation of attention and MLP
# Centerpiece: how well the EOS text embedding is reconstructed as we progressively mean-ablate
# (replace per-sample contribution by the dataset mean) attention layers, then MLP layers.
def reconstruction_cosine(rec, target):
    rec = rec / rec.norm(dim=-1, keepdim=True)
    target = target / target.norm(dim=-1, keepdim=True)
    return (rec * target).sum(-1).mean().item()

target = final_embeddings_texts_decomp  # exact sum-of-components target

# Sanity: the decomposition reproduces the model's own encode_text output (small sample)
with torch.no_grad():
    _sample = list(corpus_texts[:256])
    true_clip = model.encode_text(tokenizer(_sample).to(device), normalize=True).float()
print("Decomposition validity (sum of components vs. encode_text):",
      round(reconstruction_cosine(final_embeddings_texts_decomp[:len(_sample)], true_clip), 4))

# Mean-ablate attention, accumulating from the first layer upward
cos_attn = []
for L in range(nr_layers_ + 1):
    abl = (torch.mean(no_heads_attentions_[:, :L], axis=0).sum(0)
           + no_heads_attentions_[:, L:].sum(1)
           + mlps_.sum(axis=1))
    cos_attn.append(reconstruction_cosine(abl, target))

# Mean-ablate MLPs, accumulating from the first layer upward
cos_mlp = []
for L in range(mlps_.shape[1] + 1):
    abl = (attns_.sum(axis=(1, 2))
           + torch.mean(mlps_[:, :L], axis=0).sum(0)
           + mlps_[:, L:].sum(1))
    cos_mlp.append(reconstruction_cosine(abl, target))

plt.figure(figsize=(8, 5))
plt.plot(range(len(cos_attn)), [1 - c for c in cos_attn], marker='o', label="mean-ablate attention")
plt.plot(range(len(cos_mlp)),  [1 - c for c in cos_mlp],  marker='s', label="mean-ablate MLP")
plt.xlabel("Accumulated mean-ablated layers (from layer 0)")
plt.ylabel("EOS reconstruction error  (1 - cosine)")
plt.title(f"Text encoder EOS reconstruction error vs. mean-ablation ({model_name})")
plt.legend(); plt.grid(True); plt.show()


# Classification accuracy of the text-encoder reconstruction

In [ ]:
## Accuracy of the reconstructed text embedding: compare against TEXT or IMAGES
# compare_against options:
#   "text"  -> samples = reconstructed class sentences; weights = class-prompt classifier_;
#              correct if argmax == labels_ (text class). One sample per sentence (as before).
#   "image" -> weights = per-class mean of the reconstruction (class prototypes); samples =
#              final_embeddings_images; correct if argmax == image_labels_ (zero-shot over all images).
compare_against = "image"
assert compare_against in {"text", "image"}, "compare_against must be one of {'text', 'image'}"

@torch.no_grad()
def class_weights(rec):
    """Per-class prototype (mean over each class's reconstructed sentences), L2-normalized -> [C, d]."""
    C = int(labels_.max().item()) + 1
    W = torch.stack([rec[labels_ == c].mean(0) for c in range(C)])
    return W / W.norm(dim=-1, keepdim=True)

@torch.no_grad()
def evaluate(rec, label="reconstruction"):
    if compare_against == "text":
        return test_accuracy(rec @ classifier_, labels_, label=label)[0]
    imgs = final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)
    pred = (imgs @ class_weights(rec).T).argmax(1)
    acc = (pred == image_labels_).float().mean().item() * 100
    print(f"For the approach {label}, the accuracy is: {acc:3f}%")
    return acc

# Baseline (full decomposition) + accuracy vs. accumulated attention mean-ablation
baseline = attns_.sum(axis=(1, 2)) + mlps_.sum(axis=1)          # [N, d]
evaluate(baseline, label=f"Text baseline (vs {compare_against})")

accs = []
for L in range(nr_layers_ + 1):
    abl = (torch.mean(no_heads_attentions_[:, :L], axis=0).sum(0)
           + no_heads_attentions_[:, L:].sum(1)
           + mlps_.sum(axis=1))
    accs.append(evaluate(abl, label=f"Attn mean-ablated up to layer {L}"))

plt.figure(figsize=(8, 5))
plt.plot(range(len(accs)), accs, marker="o", label=f"{model_name} (vs {compare_against})")
plt.xlabel("Accumulated mean-ablated attention layers (from layer 0)")
plt.ylabel(f"Accuracy (compare_against={compare_against})")
plt.title(f"Text-encoder EOS accuracy vs. mean-ablation ({model_name})")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
## Completeness: reconstruction quality vs. number of principal components kept
data_all = sort_data_by(get_data(attention_dataset, -1, skip_final=True), "strength_abs", descending=True)
xs, ys = [], []
for k in [1, 5, 10, 20, 40, 80, 160, 320]:
    if k > len(data_all):
        break
    rec = reconstruct_all_embeddings_mean_ablation_pcs(
        top_data(data_all, k), mlps_, attns_, attns_, nr_layers_, nr_heads_, last_,
        ratio=-1, ablation=True, mean_ablate_all=True).to(device)
    ys.append(reconstruction_cosine(rec, target)); xs.append(k)

plt.figure(figsize=(8, 5))
plt.plot(xs, ys, marker='o', label=model_name)
plt.xlabel("Number of principal components kept (rest mean-ablated)")
plt.ylabel("Reconstruction cosine to true EOS embedding")
plt.title("Text-encoder completeness: PCs kept vs. EOS reconstruction")
plt.legend(); plt.grid(True); plt.show()


# Spatial decomposition: which input tokens drive a component (per-source-token c_{i,l,h})

In [ ]:
## On-the-fly token attribution (spatial decomposition of a single text)
# Per-source-token contribution c_{i,l,h} to the EOS embedding for a chosen (layer, head),
# using attn_method="head". Registers a one-off capture hook and removes it afterwards
# (so it never conflicts with the summed-component pipeline).
@torch.no_grad()
def token_attribution(text, layer, head, topn=15):
    toks = tokenizer([text]).to(device)
    eos = toks.argmax(dim=-1).item()
    captured = {}
    def grab(ret):
        captured['out'] = ret.detach()
        return ret
    h = model.transformer.resblocks[layer].attn.hook
    h.register("out.post", grab)
    try:
        model.encode_text(toks, normalize=False, attn_method="head")  # [.., n, m, h, d] at out.post
    finally:
        try: h.unregister("out.post", grab)
        except Exception: pass
    out = captured['out']                       # [1, n_query, m_source, h, d]
    contrib = out[0, eos, :eos + 1, head, :]     # [m, d]: source-token contributions to EOS
    mags = contrib.norm(dim=-1).cpu().numpy()
    ids = toks[0][:eos + 1].tolist()
    try:
        labels = [tokenizer.decode([i]) if hasattr(tokenizer, "decode") else str(i) for i in ids]
    except Exception:
        labels = [str(i) for i in ids]

    order = np.argsort(mags)[::-1][:topn]
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(order)), mags[order])
    plt.xticks(range(len(order)), [labels[i] for i in order], rotation=60, ha='right')
    plt.ylabel("||contribution|| to EOS"); plt.title(f"Layer {layer}, head {head}: '{text}'")
    plt.tight_layout(); plt.show()
    return labels, mags

_ = token_attribution("A photo of a polar bear on the ice.", layer=nr_layers_ - 1, head=0)
